# Lippmann Schwinger Solver

!!! note "Important points covered in this example"
      - Using VDIM for volume integral operators
      - Solving a volume integral equation

In [1]:
using Inti

## Problem definition

In this example we will solve the Lippmann Schwinger Volume Integral Equation
in a domain $\Omega$:
$$
  \begin{align}
      u + k^2 \mathcal{V}_k[(1 - \eta) u] &= u^{\textit{inc}}  \quad \text{in } \Omega \\
  \end{align}
$$
where $u^{\textit{inc}} : \Omega \to \mathbb{C}$ is a given free-space Helmholtz
solution.

In [2]:
interpolation_order = 2 # `interpolation_order` corresponds to `n` in the VDIM paper
qorder = Inti.Triangle_VR_interpolation_order_to_quadrature_order(interpolation_order)

k₁ = 4π
k₂ = 2π
λ₁ = 2π / k₁
λ₂ = 2π / k₂
meshsize   = min(λ₁,λ₂) / 7
nothing # hide

!!! note "Refraction Index Perturbation"
      A VIE with piecewise-constant refraction index perturbation $η$ can be
      verified against a BIE formulation.  Generally, we will want to use a
      VIE formulation for variable media e.g. `η = (x) -> 1 +
      .7*exp(-40*(x[1]^2 + x[2]^2))`.

In [3]:
η = (x) -> (k₂ / k₁)^2
nothing # hide

## Meshing

We now create the required meshes and quadratures for both $\Omega$ and $\Gamma$:

In [4]:
using Gmsh # this will trigger the loading of Inti's Gmsh extension

function gmsh_disk(; name, meshsize, order = 1, center = (0, 0), paxis = (1, 1))
    try
        gmsh.initialize()
        gmsh.option.setNumber("General.Terminal", 0)
        gmsh.model.add("circle-mesh")
        gmsh.option.setNumber("Mesh.MeshSizeMax", meshsize)
        gmsh.option.setNumber("Mesh.MeshSizeMin", meshsize)
        gmsh.model.occ.addDisk(center[1], center[2], 0, paxis[1], paxis[2])
        gmsh.model.occ.synchronize()
        gmsh.model.mesh.generate(2)
        gmsh.model.mesh.setOrder(order)
        gmsh.write(name)
    finally
        gmsh.finalize()
    end
end

name = joinpath(@__DIR__, "disk.msh")
gmsh_disk(; meshsize, order = 2, name)

Ω, msh = Inti.import_mesh_from_gmsh_file(name; dim = 2)
Γ = Inti.boundary(Ω)

Ωₕ = view(msh, Ω)
Γₕ = view(msh, Γ)

Info    : Reading '/home/lfaria/runner-integral-equations/_work/Inti.jl/Inti.jl/docs/src/examples/generated/disk.msh'...
Info    : 3 entities
Info    : 3029 nodes
Info    : 1559 elements
Info    : Done reading '/home/lfaria/runner-integral-equations/_work/Inti.jl/Inti.jl/docs/src/examples/generated/disk.msh'
┌ Warning: overwriting an existing entity with the same (dim,tag)=(0,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90
┌ Warning: overwriting an existing entity with the same (dim,tag)=(1,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90
┌ Warning: overwriting an existing entity with the same (dim,tag)=(2,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90


Inti.SubMesh{2, Float64} containing:
	 88 elements of type Inti.LagrangeElement{Inti.ReferenceHyperCube{1}, 3, StaticArraysCore.SVector{2, Float64}}

Use VDIM with the Vioreanu-Rokhlin quadrature rule

In [5]:
Q = Inti.VioreanuRokhlin(; domain = :triangle, order = qorder);
dict = Dict(E => Q for E in Inti.element_types(Ωₕ))
Γₕ_quad = Inti.Quadrature(Γₕ; qorder)
Ωₕ_quad = Inti.Quadrature(Ωₕ, dict)

8820-element Inti.Quadrature{2, Float64}:
 Inti.QuadratureNode{2, Float64}([0.7085523185418651, -0.6957117234492228], 0.00029959582378950747, nothing)
 Inti.QuadratureNode{2, Float64}([0.7438315432710618, -0.6578191617395827], 0.00030066272014903474, nothing)
 Inti.QuadratureNode{2, Float64}([0.6834925452101541, -0.6422127025301838], 0.0002965021190085747, nothing)
 Inti.QuadratureNode{2, Float64}([0.7135565334436431, -0.6510673764355404], 0.0006066568092388487, nothing)
 Inti.QuadratureNode{2, Float64}([0.6971227859901199, -0.6687184669649734], 0.0006056471261895217, nothing)
 Inti.QuadratureNode{2, Float64}([0.7254531908708148, -0.6761961849203815], 0.0006095846112216898, nothing)
 Inti.QuadratureNode{2, Float64}([0.9269994063685345, 0.3560933246100717], 0.0002980934947338548, nothing)
 Inti.QuadratureNode{2, Float64}([0.907186639451076, 0.4039255752103367], 0.0002978124123797406, nothing)
 Inti.QuadratureNode{2, Float64}([0.866455585020041, 0.3601204063896579], 0.0002943258005961014

## Volume Integral Operators and Volume Integral Equations

In [6]:
using FMMLIB2D

pde = Inti.Helmholtz(; dim = 2, k = k₁)

Δu + k² u = 0

With quadratures constructed on the volume, we can define a discrete approximation
to the volume integral operator $\mathcal{V}$ using VDIM.

In [7]:
V_d2d = Inti.volume_potential(;
    pde,
    target = Ωₕ_quad,
    source = Ωₕ_quad,
    compression = (method = :fmm, tol = 1e-7),
    correction = (method = :dim, interpolation_order)
)

using LinearAlgebra
using LinearMaps

using SpecialFunctions
uⁱ = (x) -> exp(im * k₁ * x[2]) # plane-wave incident field
rhs = map(Ωₕ_quad) do q
    x = q.coords
    return uⁱ(x)
end

8820-element Vector{ComplexF64}:
  -0.7761833301703072 - 0.6305072862114536im
  -0.4008255293553406 - 0.916154405663702im
 -0.21461848830071084 - 0.9766979596986561im
 -0.32174538127272384 - 0.9468262299015958im
  -0.5221606600892983 - 0.8528471404977027im
  -0.5998792931374709 - 0.8000905159198478im
 -0.23535935194499724 - 0.9719084192721199im
  0.35553789229866795 - 0.9346618678109323im
  -0.1858948299949908 - 0.9825696474963662im
  0.08081209162412846 - 0.9967293543622228im
                      ⋮
  -0.9963280570569449 + 0.08561777105912662im
  -0.9644699925952754 + 0.26419241734627674im
  -0.9964573317017412 + 0.08409985789432771im
  0.39077911940621873 - 0.9204844810403379im
    0.336851605701448 - 0.9415577495493074im
 -0.03248412251525768 - 0.9994722516330374im
   0.1604482855076432 - 0.9870442480849873im
   0.1870113255606387 - 0.9823577576993286im
  0.35535645285038253 - 0.9347308657670367im

The full VIO incorporates scalar point multiplication using the contrast function η, implemented as a composition of `LinearMap`

In [8]:
refr_map_d = map(Ωₕ_quad) do q
    x = q.coords
    return 1 - η(x)
end
apply_refr!(y, x) = y .= refr_map_d .* x
Lη = LinearMap{ComplexF64}(apply_refr!, length(refr_map_d), length(refr_map_d))
L = I + k₁^2 * V_d2d * Lη

8820×8820 LinearMaps.LinearCombination{ComplexF64} with 2 maps:
  8820×8820 LinearMaps.UniformScalingMap{Bool} with scaling factor: true
  8820×8820 LinearMaps.ScaledMap{ComplexF64} with scale: 157.91367041742973 of
    8820×8820 LinearMaps.CompositeMap{ComplexF64} with 2 maps:
      8820×8820 LinearMaps.FunctionMap{ComplexF64,true}(apply_refr!; issymmetric=false, ishermitian=false, isposdef=false)
      8820×8820 LinearMaps.LinearCombination{ComplexF64} with 2 maps:
        8820×8820 LinearMaps.FunctionMap{ComplexF64,true}(#7; issymmetric=false, ishermitian=false, isposdef=false)
        8820×8820 LinearMaps.WrappedMap{ComplexF64} of
          8820×8820 SparseArrays.SparseMatrixCSC{ComplexF64, Int64} with 52920 stored entries

The unknown volumetric field $u$:

In [9]:
using IterativeSolvers
u, hist =
    gmres(L, rhs; log = true, abstol = 1e-7, verbose = false, restart = 200, maxiter = 200)
@show hist

𝒱 = Inti.IntegralPotential(Inti.SingleLayerKernel(pde), Ωₕ_quad)

hist = Converged after 102 iterations.


Inti.IntegralPotential{Inti.SingleLayerKernel{ComplexF64, Inti.Helmholtz{2, Float64}}, Inti.Quadrature{2, Float64}}(Inti.SingleLayerKernel{ComplexF64, Inti.Helmholtz{2, Float64}}(Δu + k² u = 0),  Quadrature with 8820 quadrature nodes)

The representation formula gives the solution in $\R^2 \setminus \Omega$:

In [10]:
uˢ = (x) -> uⁱ(x) - k₁^2 * 𝒱[refr_map_d .* u](x)
nothing # hide

To visualize the solution using Gmsh, let's query it at the triangle vertices  in $\Omega$

In [11]:
solₕ_nodes = Inti.quadrature_to_node_vals(Ωₕ_quad, real(-u))

gmsh.initialize()
Inti.write_gmsh_model(msh)
Inti.write_gmsh_view!(Ωₕ, solₕ_nodes; name="LS solution")
"-nopopup" in ARGS || gmsh.fltk.run()
gmsh.finalize()
nothing # hide

[ Info: Inti.LagrangeElement{Inti.ReferenceSimplex{2}, 6, StaticArraysCore.SVector{2, Float64}}


---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*